# 2.0 Decision Tree Classification

## 1. Core Intuition: The "20 Questions" Game
A **Decision Tree** is a supervised machine learning model that makes predictions by breaking down data into successive yes/no questions—exactly like a game of *20 Questions* or a flowchart.

### Tree Anatomy
- **Root Node**: The top-most decision rule where the entire dataset starts.
- **Internal Decision Nodes**: Subsequent splits testing specific feature thresholds (e.g., `Age <= 44.5`).
- **Leaf Nodes (Terminal Nodes)**: The final outcome or class prediction (`Purchased: Yes` or `No`).

---

## 2. How Does the Tree Decide Where to Split? (Gini Impurity)
Decision Trees evaluate every possible split across all features and choose the one that produces the purest subsets.

**Gini Impurity Formula**:
$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

- **$Gini = 0.0$ (Pure)**: Every single observation in the node belongs to the exact same class (e.g., 100% Purchased).
- **$Gini = 0.5$ (Maximum Impurity)**: The node is evenly split (50% No Purchase, 50% Purchased).
- The tree greedily selects the split that maximizes **Information Gain** (i.e., minimizes resulting Gini impurity).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Step 1: Loading Dataset & Splitting

> **Crucial Insight**: Do Decision Trees require Feature Scaling?
>
> **NO!** Decision Trees are scale-invariant. A split condition like `Age <= 44.5` depends strictly on the ordering of values, not their absolute scale. Unlike kNN, Decision Trees work equally well on raw or scaled features.

In [ ]:
# Load dataset
df = pd.read_csv('datasets/social_network_ads.csv')

feature_names = ['Age', 'EstimatedSalary']
target_name = 'Purchased'

X = df[feature_names].values
y = df[target_name].values

# 75% Train, 25% Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Step 2: Training the Decision Tree Classifier
We constrain `max_depth=3` to keep the model interpretable and prevent immediate overfitting.

In [ ]:
# Initialize tree with max_depth=3
dt_classifier = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    random_state=42
)

# Train model
dt_classifier.fit(X_train, y_train)

# Predictions
y_pred = dt_classifier.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Decision Tree Test Accuracy (max_depth=3): {acc * 100:.2f}%")

## Step 3: Visualizing the Tree Flowchart
Unlike black-box models (like Deep Neural Networks), Decision Trees are **white-box models**—we can print the exact flowchart of rules used by the algorithm!

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(
    dt_classifier,
    feature_names=['Age', 'Estimated Salary'],
    class_names=['No Purchase', 'Purchased'],
    filled=True,
    rounded=True,
    fontsize=11
)
plt.title('Trained Decision Tree Visualization (max_depth=3)', fontsize=16, fontweight='bold', pad=15)
plt.show()

## Step 4: Model Evaluation Metrics
Let's inspect the confusion matrix and precision/recall trade-offs.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Pred: No Purchase (0)', 'Pred: Purchased (1)'],
            yticklabels=['Actual: No Purchase (0)', 'Actual: Purchased (1)'])
plt.title('Decision Tree Confusion Matrix', fontsize=13, fontweight='bold')
plt.show()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Purchase (0)', 'Purchased (1)']))

## Step 5: Feature Importance
Which feature mattered more for driving purchases? `Age` or `EstimatedSalary`?
Decision Trees compute feature importance based on total reduction in Gini impurity brought by each feature.

In [ ]:
importances = pd.Series(dt_classifier.feature_importances_, index=['Age', 'EstimatedSalary'])

plt.figure(figsize=(7, 3))
importances.plot(kind='barh', color='#27ae60')
plt.title('Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Normalized Importance Score', fontsize=11)
plt.xlim(0, 1)
plt.show()

print("Importance values:")
print(importances.round(4))

## Step 6: The Overfitting Dilemma & Pruning (`max_depth`)

What happens if we let the tree grow without restriction (`max_depth=None`)?
- It keeps splitting until every leaf node has $Gini = 0$ (perfect training purity).
- **Result**: The tree memorizes noise, outliers, and training quirks, and fails on unseen test data.

Let's sweep `max_depth` from 1 to 10 and compare Train vs. Test Accuracy.

In [ ]:
depths = range(1, 11)
train_scores = []
test_scores = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test, tree.predict(X_test)))

plt.figure(figsize=(9, 4.5))
plt.plot(depths, train_scores, marker='o', label='Training Accuracy (Overfitting as depth increases)', color='#e74c3c')
plt.plot(depths, test_scores, marker='s', label='Testing Accuracy (Generalization)', color='#2980b9')
plt.title('Tree Depth vs. Accuracy: Identifying Overfitting', fontsize=14, fontweight='bold')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Accuracy Score', fontsize=12)
plt.xticks(depths)
plt.legend()
plt.grid(True)
plt.show()

## 3. Comparison Matrix: kNN vs. Decision Trees

| Characteristic | k-Nearest Neighbors (kNN) | Decision Trees |
| :--- | :--- | :--- |
| **Learning Paradigm** | Instance-based / Lazy learner | Model-based / Eager learner |
| **Training Cost** | $O(1)$ (almost zero training work) | $O(N \log N \times D)$ (computes best splits) |
| **Inference Cost** | High ($O(N \times D)$ distance calculations) | Extremely Fast ($O(\text{depth})$ IF-ELSE checks) |
| **Feature Scaling** | **Mandatory** (distance-sensitive) | **Not Required** (scale invariant) |
| **Interpretability** | Moderate (based on neighbors) | Very High (direct visual flowchart) |
| **Primary Risk** | High memory usage, sensitive to noise | Overfitting on deep trees |